In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
import seaborn as sns
import math
from cmdstanpy import CmdStanModel

/opt/miniconda3/envs/genetics_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
import cmdstanpy
cmdstanpy.install_cmdstan(overwrite = True)

CmdStan install directory: /Users/qichen/.cmdstan
Installing CmdStan version: 2.37.0
Download successful, file: /var/folders/gm/vzcskl_s1wzfqy1p36xcp32h0000gn/T/tmpbims3tvn
Extracting distribution
Unpacked download as cmdstan-2.37.0
Building version cmdstan-2.37.0, may take several minutes, depending on your system.
Installed cmdstan-2.37.0
Test model compilation


True

In [2]:
snp = pd.read_json('../json_snp.json')
snp['d_mean'] = snp['d_mean']*100
snp['d_var'] = snp['d_var']*100
print(snp)

    group    d_mean     d_var  adjusted_factor
0  [0, 0]  0.817305  0.023009          0.18755
1  [0, 1] -0.010856  0.016651          0.18755
2  [0, 2] -0.090904  0.019700          0.18755
3  [0, 3] -0.715544  0.027234          0.18755
4  [1, 1]  0.679156  0.021560          0.18755
5  [1, 2]  0.140838  0.017736          0.18755
6  [1, 3] -0.809138  0.030801          0.18755
7  [2, 2]  0.804949  0.023561          0.18755
8  [2, 3] -0.854883  0.029888          0.18755
9  [3, 3]  2.379564  0.062573          0.18755


In [3]:
json_snp = snp.to_dict(orient = 'list')
json_snp['N_obs'] = len(snp)
json_snp['N_pop'] = 4
json_snp['ancestral_T'] = 800

In [7]:
snp_model = CmdStanModel(stan_file="snp_alone_with_weights.stan")

15:49:03 - cmdstanpy - INFO - compiling stan file /Users/qichen/github/admix_stan/new_pipeline/snp_alone_with_weights.stan to exe file /Users/qichen/github/admix_stan/new_pipeline/snp_alone_with_weights
15:49:09 - cmdstanpy - INFO - compiled model executable: /Users/qichen/github/admix_stan/new_pipeline/snp_alone_with_weights


In [12]:
fit = snp_model.sample(
    data=json_snp,
    chains=16, parallel_chains=16,
    iter_warmup=1500, iter_sampling=1500,
    adapt_delta=0.99,        # 0.95 → 0.99 (or 0.999 if needed)
    max_treedepth=12
)

21:26:54 - cmdstanpy - INFO - CmdStan start processing
chain 1:   0%|          | 0/3000 [00:00<?, ?it/s, (Warmup)]































































































































































































chain 1:   7%|▋         | 200/3000 [00:00<00:05, 542.72it/s, (Warmup)]












































































































































chain 1:  10%|█         | 300/3000 [00:00<00:06, 421.51it/s, (Warmup)]







































































































chain 1:  13%|█▎        | 400/3000 [00:00<00:05, 472.60it/s, (Warmup)]



















































































chain 1:  17%|█▋        | 500/3000 [00:01<00:05, 469.66it/s, (Warmup)]





































































































































chain 1:  20%|██        | 600/3000 [00:01<00:05, 464.46it/s, (Warmup)]





21:27:02 - cmdstanpy - INFO - CmdStan done processing.


In [13]:
fit.summary()

,Mean,MCSE,StdDev,MAD,5%,50%,95%,ESS_bulk,ESS_tail,ESS_bulk/s,R_hat
lp__,-21.267100,2.825800e-02,2.178530,2.051320,-25.324100,-20.939700,-18.354700,6170.11,9721.34,129.4340,1.00192
c1,0.005937,2.482390e-05,0.001617,0.001139,0.002300,0.006405,0.007675,5918.44,3828.92,124.1540,1.00272
c2,0.001830,2.511390e-05,0.001660,0.001206,0.000133,0.001314,0.005524,5852.01,4302.07,122.7610,1.00259
c3,0.004654,3.049460e-05,0.001875,0.001822,0.000927,0.005117,0.006999,4060.40,4212.83,85.1772,1.00392
c4,0.005660,3.069570e-05,0.001901,0.001835,0.003287,0.005193,0.009393,4180.42,5020.76,87.6950,1.00413
c5,0.019127,1.223670e-04,0.011177,0.014331,0.001771,0.019108,0.036492,7932.89,7574.26,166.4130,1.00240
c6,0.019368,1.222520e-04,0.011174,0.014376,0.001963,0.019417,0.036683,7885.18,7022.40,165.4120,1.00223
c7,0.006024,9.132830e-06,0.000645,0.000704,0.004952,0.006034,0.007039,5010.72,9970.67,105.1130,1.00321
f,0.359628,2.897400e-03,0.182487,0.230281,0.081061,0.354158,0.650068,3974.38,7135.43,83.3727,1.00420
a_11,0.026894,1.222740e-04,0.011183,0.014358,0.009513,0.026885,0.044274,7955.86,7617.25,166.8950,1.00237
